In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (20,10)

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
from sklearn import datasets, linear_model, metrics
import pandas as pd

In [82]:
# download csv from url
#url = 'https://archives.nseindia.com/content/indices/ind_nifty100list.csv' # Nifty 100 stocks
#nifty100 = pd.read_csv(url)
#nifty100= nifty100.to_csv('ind_nifty100list.csv', index=False)

#symbol= pd.read_csv("ind_nifty100list.csv")  # read csv file
#symbol["ticker"] = symbol["Symbol"] + ".NS"
#ticker= symbol['ticker'].values.tolist() # convert to list
#sd = yf.download(ticker, interval='1d')['Close'] # download data from yahoo finance
#sd  = yf.download(ticker, start="2017-01-01", end="2024-09-30")
#idx  = yf.download("^NSEI", start="2017-01-01", end="2024-09-30")

In [2]:
symbols= ['RELIANCE.NS','HDFCBANK.NS','ICICIBANK.NS','INFY.NS','ITC.NS','TCS.NS','HINDUNILVR.NS','SBIN.NS','BHARTIARTL.NS','^NSEI']
sd = yf.download(symbols, interval='1d')['Close']
sd.reset_index(inplace=True)
# rename column ['Date'] to ['date']
sd.rename(columns={'^NSEI':'Nifty50'}, inplace=True)
# reaarange columns
sd = sd[['Date','RELIANCE.NS','HDFCBANK.NS','ICICIBANK.NS','INFY.NS','ITC.NS','TCS.NS','HINDUNILVR.NS','SBIN.NS','BHARTIARTL.NS','Nifty50']]
symbols.remove('^NSEI')
symbols.append('Nifty50')
sd


[*********************100%***********************]  10 of 10 completed


,Date,RELIANCE.NS,HDFCBANK.NS,ICICIBANK.NS,INFY.NS,ITC.NS,TCS.NS,HINDUNILVR.NS,SBIN.NS,BHARTIARTL.NS,Nifty50
0,1996-01-01,14.691803,2.980000,NaN,0.796679,5.583333,NaN,61.805000,18.823240,NaN,NaN
1,1996-01-02,14.577553,2.975000,NaN,0.793457,5.372222,NaN,62.465000,18.224106,NaN,NaN
2,1996-01-03,14.688232,2.985000,NaN,0.798828,5.200000,NaN,62.095001,17.738192,NaN,NaN
3,1996-01-04,14.552561,2.965000,NaN,0.793554,5.297777,NaN,62.099998,17.676863,NaN,NaN
4,1996-01-05,14.452592,2.960000,NaN,0.784179,5.202222,NaN,62.000000,17.577793,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
7220,2024-09-27,3052.350098,1752.650024,1306.599976,1906.750000,522.700012,4308.700195,2966.250000,802.650024,1734.599976,26178.949219
7221,2024-09-30,2953.149902,1732.050049,1273.000000,1875.599976,518.150024,4268.500000,2958.300049,787.900024,1709.550049,25810.849609
7222,2024-10-01,2929.649902,1726.199951,1274.400024,1904.349976,516.200012,4287.899902,2923.750000,796.950012,1698.699951,25796.900391
7223,2024-10-03,2813.949951,1682.000000,1256.349976,1893.400024,512.750000,4232.750000,2893.350098,794.099976,1673.449951,25250.099609


In [3]:
sd1=sd.dropna()
sd1


,Date,RELIANCE.NS,HDFCBANK.NS,ICICIBANK.NS,INFY.NS,ITC.NS,TCS.NS,HINDUNILVR.NS,SBIN.NS,BHARTIARTL.NS,Nifty50
3023,2007-09-17,463.635254,122.614998,162.754547,225.600006,60.283333,249.412506,215.899994,155.265793,366.991943,4494.649902
3024,2007-09-18,470.435608,123.114998,168.100006,225.568756,60.083332,250.612503,214.550003,159.922058,375.170868,4546.200195
3025,2007-09-19,496.916962,132.429993,177.009094,231.449997,62.450001,255.762497,217.000000,166.974884,399.640015,4732.350098
3026,2007-09-20,500.665741,132.619995,175.836365,225.050003,64.699997,250.512497,214.199997,166.017212,402.794403,4747.549805
3027,2007-09-21,521.272522,131.845001,175.645447,228.237503,63.633331,254.449997,219.399994,170.522522,415.141632,4837.549805
...,...,...,...,...,...,...,...,...,...,...,...
7220,2024-09-27,3052.350098,1752.650024,1306.599976,1906.750000,522.700012,4308.700195,2966.250000,802.650024,1734.599976,26178.949219
7221,2024-09-30,2953.149902,1732.050049,1273.000000,1875.599976,518.150024,4268.500000,2958.300049,787.900024,1709.550049,25810.849609
7222,2024-10-01,2929.649902,1726.199951,1274.400024,1904.349976,516.200012,4287.899902,2923.750000,796.950012,1698.699951,25796.900391
7223,2024-10-03,2813.949951,1682.000000,1256.349976,1893.400024,512.750000,4232.750000,2893.350098,794.099976,1673.449951,25250.099609


In [7]:
def momentum_ratio(sd1): # function to calculate momentum ratio
    for i in sd1.columns:
        ret_daily = sd1[i].pct_change(periods=1) # percent change over daily trading days
        stdev_5 = ret_daily.rolling(window=5).std() # standard deviation over 5 trading days
        avg_ret = ret_daily.rolling(window=5).median()
        sd1[i+'_Year_momentum'] = avg_ret / stdev_5 
        sd1.dropna(inplace=True) # drop rows with NaN values
    return sd1

momentum_df = momentum_ratio(sd1)

TypeError: cannot perform __truediv__ with this index type: DatetimeArray